# 🔴 Solution: Rotary Position Embedding (RoPE)（面试版）

## 📋 题目描述

**难度：** Medium

实现旋转位置编码（RoPE）。

RoPE 通过成对旋转查询和键向量来编码位置信息，利用点积性质实现相对位置感知。

**签名:** `apply_rope(q, k) -> (Tensor, Tensor)`

**参数:**
- `q` — 查询张量 (B, S, D)
- `k` — 键张量 (B, S, D)

**返回:** 旋转后的 (q, k) 元组，形状不变

**约束:**
- 按奇偶对分割，使用 `angles = pos * 1/(10000^(2i/D))` 旋转
- 必须保持向量范数不变
- 点积应仅依赖于相对位置

**提示：** `freqs = 1 / 10000^(2i/D)`；`angles = pos * freqs` → `cos_a`、`sin_a`；拆分 `x[..., 0::2]`、`x[..., 1::2]`；旋转后 stack → flatten。

In [ ]:
import torch
import math

In [ ]:
# ✅ INTERVIEW — 手写 RoPE（不使用任何位置编码库）

# ============================================================
# 数学背景：
#   RoPE 对每对相邻维度 (2i, 2i+1) 做二维旋转
#   旋转角度 = position * frequency
#   频率 = 1 / (10000 ^ (2i/D))，随维度指数衰减
#
#   旋转矩阵：
#   [cos θ  -sin θ] [q_2i    ]   [q_2i cos θ - q_{2i+1} sin θ]
#   [sin θ   cos θ] [q_{2i+1}] = [q_2i sin θ + q_{2i+1} cos θ]
#
#   性质：旋转后的点积只依赖相对位置 (pos_q - pos_k)
#   因为 R(θ1)^T @ R(θ2) = R(θ2 - θ1)
# ============================================================

def apply_rope(q, k):
    """
    手写实现旋转位置编码（RoPE）。

    参数:
        q: Tensor, shape = (B, S, D) — 查询张量
        k: Tensor, shape = (B, S, D) — 键张量

    返回:
        (q_rotated, k_rotated) — 旋转后的查询和键，形状不变
    """

    # ---- Step 1: 获取形状参数 ----
    # B = batch size, S = 序列长度, D = 特征维度
    # D 必须是偶数（因为要两两配对旋转）
    B, S, D = q.shape

    # ---- Step 2: 生成位置索引 [0, 1, 2, ..., S-1] ----
    # unsqueeze(1) 把形状从 (S,) 变成 (S, 1)
    # float() 转浮点，方便后续数学运算
    #
    # 例：S=4 -> pos = [[0.], [1.], [2.], [3.]]，shape = (4, 1)
    pos = torch.arange(S, device=q.device).unsqueeze(1).float()

    # ---- Step 3: 生成维度索引 [0, 2, 4, ..., D-2] ----
    # 只取偶数索引，因为每对 (2i, 2i+1) 共享一个频率
    # D//2 个频率值
    #
    # 例：D=8 -> dim = [0., 2., 4., 6.]，shape = (4,)
    dim = torch.arange(0, D, 2, device=q.device).float()

    # ---- Step 4: 计算频率 ----
    # 公式：freq_i = 1 / (10000 ^ (2i / D))
    #
    # 为什么用 10000？
    #   这是 LLaMA/RoFormer 的标准选择
    #   10000^(2i/D) 在 i=0 时为 1（最高频），i=D/2 时为 10000（最低频）
    #   使得不同维度编码不同频率的位置信息
    #
    # 例：D=8 时
    #   dim/8 = [0, 0.25, 0.5, 0.75]
    #   10000^(dim/8) = [1, 5.62, 31.62, 177.83]
    #   freqs = [1, 0.178, 0.0316, 0.00562]
    freqs = 1.0 / (10000.0 ** (dim / D))

    # ---- Step 5: 计算旋转角度 = 位置 * 频率 ----
    # pos: (S, 1) * freqs: (D/2,) -> 广播 -> angles: (S, D/2)
    #
    # 例：pos=[[0],[1],[2],[3]], freqs=[1, 0.178]
    #   angles = [[0, 0],
    #             [1, 0.178],
    #             [2, 0.356],
    #             [3, 0.534]]
    angles = pos * freqs

    # ---- Step 6: 预计算 cos 和 sin ----
    # cos_a 和 sin_a 形状都是 (S, D/2)
    # 后面会用它们做旋转
    cos_a = torch.cos(angles)
    sin_a = torch.sin(angles)

    # ---- Step 7: 定义旋转函数 ----
    # 这是 RoPE 的核心操作
    #
    # 对输入 x (B, S, D)：
    #   x[..., 0::2] 取偶数维度 -> (B, S, D/2)  <- 对应旋转公式的 q_2i
    #   x[..., 1::2] 取奇数维度 -> (B, S, D/2)  <- 对应旋转公式的 q_{2i+1}
    #
    # 切片 [start:stop:step] 语法：
    #   0::2 = 从 0 开始，步长 2 -> [0, 2, 4, ...]
    #   1::2 = 从 1 开始，步长 2 -> [1, 3, 5, ...]
    #
    # 旋转公式：
    #   new_2i     = x_2i * cos - x_{2i+1} * sin
    #   new_{2i+1} = x_2i * sin + x_{2i+1} * cos
    #
    # 然后 stack 拼回 (B, S, D/2, 2)，最后 flatten(-2) 展平为 (B, S, D)
    #
    # 为什么用 flatten(-2) 而不是 reshape？
    #   flatten(-2) 把最后两维合并，语义更清晰，且保证连续内存
    def rotate(x):
        x1 = x[..., 0::2]  # 偶数维度 (B, S, D/2)
        x2 = x[..., 1::2]  # 奇数维度 (B, S, D/2)
        # cos_a, sin_a: (S, D/2)，会自动广播到 (B, S, D/2)
        return torch.stack([
            x1 * cos_a - x2 * sin_a,  # 旋转后的偶数维度
            x1 * sin_a + x2 * cos_a   # 旋转后的奇数维度
        ], dim=-1).flatten(-2)  # (B, S, D/2, 2) -> (B, S, D)

    # ---- Step 8: 分别旋转 q 和 k ----
    return rotate(q), rotate(k)

In [ ]:
# Verify
torch.manual_seed(42)
q = torch.randn(2, 4, 8)
k = torch.randn(2, 4, 8)
q_rot, k_rot = apply_rope(q, k)
print("q shape:", q_rot.shape)
print("k shape:", k_rot.shape)

In [ ]:
from torch_judge import check
check('rope')

## 📝 核心思路总结

1. **本质**：RoPE 对每对相邻维度做二维旋转，旋转角度 = 位置 × 频率，频率按 `1/10000^(2i/D)` 指数衰减，使得不同维度编码不同尺度的位置信息。
2. **相对位置的魔法**：旋转后的点积 `q_rot · k_rot` 只依赖于相对位置 `(pos_q - pos_k)`，因为 `R(θ1)^T R(θ2) = R(θ2 - θ1)`，这是 RoPE 最核心的数学性质。
3. **实现要点**：用 `0::2` 和 `1::2` 切片把 D 维拆成 D/2 对，每对用 cos/sin 旋转，最后 stack + flatten 拼回。广播机制让 `(S, D/2)` 的 cos/sin 自动扩展到 `(B, S, D/2)`。
4. **面试考点**：为什么 D 必须是偶数？（两两配对）为什么用 10000？（频率范围覆盖）flatten(-2) vs reshape 的区别？（连续性保证）。